# 🎯 Mumbai Rent Prediction - Complete ML Pipeline

This notebook:
1. **Load Mumbai Data** - Process JSON data with nested structure
2. **Feature Engineering** - Extract and prepare features
3. **Train Models** - LightGBM and ANN with optimized parameters
4. **Evaluate Performance** - Compare models and analyze predictions
5. **Bias Correction** - Fix systematic errors for deployment

## 📊 Step 1: Load and Parse Mumbai Data

In [ ]:
# Installation
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn tensorflow

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import glob
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score, mean_squared_error
import lightgbm as lgb
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import tensorflow as tf

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("="*80)
print("🏙️ MUMBAI RENT PREDICTION - ML PIPELINE")
print("="*80)

# Mount Google Drive
drive.mount('/content/drive')

# Load Mumbai JSON data
def load_mumbai_data(json_path):
    """
    Load Mumbai data from JSON files with nested structure
    Expected structure: {success: true, data: {propertiesByLocality: {...}}}
    """
    all_properties = []
    
    # Handle both single file and directory patterns
    if json_path.endswith('.json'):
        json_files = [json_path]
    else:
        json_files = glob.glob(json_path)
    
    print(f"\n🔍 Found {len(json_files)} JSON file(s)\n")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            # Navigate nested structure
            if isinstance(data, list):
                # If data is already a list of properties
                for item in data:
                    if 'data' in item and 'propertiesByLocality' in item['data']:
                        localities = item['data']['propertiesByLocality']
                        for locality, properties in localities.items():
                            for prop in properties:
                                prop['locality'] = locality
                                all_properties.append(prop)
            elif isinstance(data, dict):
                # Single JSON object
                if 'data' in data and 'propertiesByLocality' in data['data']:
                    localities = data['data']['propertiesByLocality']
                    for locality, properties in localities.items():
                        for prop in properties:
                            prop['locality'] = locality
                            all_properties.append(prop)
            
            print(f"✓ Loaded: {json_file.split('/')[-1]} - {len(all_properties)} properties")
            
        except Exception as e:
            print(f"✗ Failed: {json_file.split('/')[-1]} - {e}")
            continue
    
    df = pd.DataFrame(all_properties)
    print(f"\n📊 TOTAL PROPERTIES LOADED: {len(df):,}")
    return df

# ⚠️ UPDATE THIS PATH to your Mumbai data location
MUMBAI_DATA_PATH = "/content/drive/MyDrive/Mumbai_Data/*.json"  # or specific file path

# Load data
print("\n[STEP 1] LOADING MUMBAI DATA...")
df_raw = load_mumbai_data(MUMBAI_DATA_PATH)

print(f"\n✓ Data loaded successfully")
print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")

## 🔍 Step 2: Data Inspection and Quality Check

In [ ]:
# Inspect the data structure
print("\n[STEP 2] DATA INSPECTION...")

print("\n📋 First few records:")
print(df_raw.head())

print("\n📊 Data Info:")
print(df_raw.info())

# Check price/rent statistics
if 'price' in df_raw.columns:
    print("\n💰 Rent Statistics (Before Cleaning):")
    print(f"  Count: {df_raw['price'].notna().sum():,}")
    print(f"  Min: ₹{df_raw['price'].min():,.2f}")
    print(f"  Max: ₹{df_raw['price'].max():,.2f}")
    print(f"  Mean: ₹{df_raw['price'].mean():,.2f}")
    print(f"  Median: ₹{df_raw['price'].median():,.2f}")
    
    # Quality check
    very_low = (df_raw['price'] < 1000).sum()
    very_high = (df_raw['price'] > 500000).sum()
    print(f"\n⚠️ Data Quality:")
    print(f"  Rent < ₹1,000: {very_low:,}")
    print(f"  Rent > ₹5,00,000: {very_high:,}")

# Check other key features
print("\n📋 Feature Availability:")
key_features = ['propertySize', 'bhk', 'bath', 'locality', 'floor', 'totalFloor', 'parking', 'facing']
for feature in key_features:
    if feature in df_raw.columns:
        missing = df_raw[feature].isna().sum()
        missing_pct = (missing / len(df_raw)) * 100
        unique = df_raw[feature].nunique()
        print(f"  {feature:20s}: {len(df_raw) - missing:,} valid ({100-missing_pct:.1f}%), {unique} unique values")

print("\n✓ Initial inspection complete")

## 🔧 Step 3: Feature Engineering and Data Cleaning

In [ ]:
# Clean and prepare features
print("\n[STEP 3] FEATURE ENGINEERING...")

df = df_raw.copy()

# Rename columns to match standard format
column_mapping = {
    'price': 'rent',
    'propertySize': 'propertysize',
    'bhk': 'bedroom',
    'bath': 'bathroom',
    'floor': 'floorno',
    'totalFloor': 'totalfloor',
    'facing': 'facing',
    'parking': 'parking'
}

df.rename(columns=column_mapping, inplace=True)

print(f"\nBefore cleaning: {len(df)} rows")

# 1. Clean rent/price
df = df[df['rent'].notna()]
df = df[df['rent'] > 0]
df = df[(df['rent'] >= 1000) & (df['rent'] <= 500000)]
print(f"After removing invalid rent: {len(df)} rows")

# 2. Clean property size
if 'propertysize' in df.columns:
    df = df[df['propertysize'].notna()]
    df = df[(df['propertysize'] >= 100) & (df['propertysize'] <= 10000)]
    print(f"After cleaning property size: {len(df)} rows")

# 3. Process BHK (convert "BHK2" -> 2)
if 'bedroom' in df.columns:
    def extract_bhk(bhk_val):
        if pd.isna(bhk_val):
            return np.nan
        bhk_str = str(bhk_val).upper()
        if 'BHK' in bhk_str:
            # Extract number from "BHK2", "2BHK", etc.
            import re
            numbers = re.findall(r'\d+', bhk_str)
            return int(numbers[0]) if numbers else np.nan
        try:
            return int(float(bhk_val))
        except:
            return np.nan
    
    df['bedroom'] = df['bedroom'].apply(extract_bhk)
    df = df[df['bedroom'].notna()]
    df = df[(df['bedroom'] >= 1) & (df['bedroom'] <= 10)]
    print(f"After processing BHK: {len(df)} rows")

# 4. Clean bathroom
if 'bathroom' in df.columns:
    df['bathroom'] = pd.to_numeric(df['bathroom'], errors='coerce')
    df = df[df['bathroom'].notna()]
    df = df[(df['bathroom'] >= 1) & (df['bathroom'] <= 10)]

# 5. Clean floor numbers
for col in ['floorno', 'totalfloor']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col].fillna(df[col].median(), inplace=True)
        df = df[(df[col] >= 0) & (df[col] <= 100)]

# 6. Process categorical features
if 'parking' in df.columns:
    # Convert parking to numeric: NONE=0, TWO_WHEELER=1, FOUR_WHEELER=2
    parking_map = {'NONE': 0, 'TWO_WHEELER': 1, 'FOUR_WHEELER': 2, 'BOTH': 2}
    df['parking_numeric'] = df['parking'].map(parking_map)
    df['parking_numeric'].fillna(0, inplace=True)

# 7. Keep only localities with sufficient data
if 'locality' in df.columns:
    locality_counts = df['locality'].value_counts()
    valid_localities = locality_counts[locality_counts >= 5].index
    df = df[df['locality'].isin(valid_localities)]
    print(f"After locality filtering: {len(df)} rows")

# 8. Remove outliers using IQR
Q1 = df['rent'].quantile(0.05)
Q3 = df['rent'].quantile(0.95)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df = df[(df['rent'] >= lower) & (df['rent'] <= upper)]
print(f"After removing outliers: {len(df)} rows")

# 9. Feature validation: remove unrealistic combinations
if 'propertysize' in df.columns and 'bedroom' in df.columns:
    df['size_per_bedroom'] = df['propertysize'] / (df['bedroom'] + 1)
    df = df[(df['size_per_bedroom'] >= 100) & (df['size_per_bedroom'] <= 5000)]
    df = df.drop('size_per_bedroom', axis=1)
    print(f"After feature validation: {len(df)} rows")

# 10. Create feature matrix
feature_cols = ['propertysize', 'bedroom', 'bathroom', 'locality', 
                'floorno', 'totalfloor', 'parking_numeric']

# Add facing if available
if 'facing' in df.columns:
    feature_cols.append('facing')

# Keep only available columns
available_features = [col for col in feature_cols if col in df.columns]

X = df[available_features].copy()
y = df['rent'].copy()

# Handle categorical variables
categorical_cols = ['locality', 'facing']
for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')

print(f"\n✓ Feature engineering complete")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nRent Statistics (After Cleaning):")
print(f"  Range: ₹{y.min():,.0f} - ₹{y.max():,.0f}")
print(f"  Mean: ₹{y.mean():,.0f}")
print(f"  Median: ₹{y.median():,.0f}")
print(f"\nFeatures: {list(X.columns)}")

## ✅ Step 4: Data Quality Summary

In [ ]:
print("\n" + "="*80)
print("✅ DATA QUALITY SUMMARY")
print("="*80)

print(f"\n📊 Dataset Size:")
print(f"  Total properties: {len(y):,}")
print(f"  Training set (80%): {int(len(y) * 0.8):,}")
print(f"  Test set (20%): {int(len(y) * 0.2):,}")

print(f"\n💰 Rent Distribution:")
print(f"  Min: ₹{y.min():,.0f}")
print(f"  25th percentile: ₹{y.quantile(0.25):,.0f}")
print(f"  Median: ₹{y.quantile(0.5):,.0f}")
print(f"  Mean: ₹{y.mean():,.0f}")
print(f"  75th percentile: ₹{y.quantile(0.75):,.0f}")
print(f"  Max: ₹{y.max():,.0f}")

print(f"\n🏠 Feature Statistics:")
for col in X.columns:
    if col in ['locality', 'facing']:
        print(f"  {col}: {X[col].nunique()} unique values")
    else:
        print(f"  {col}: min={X[col].min():.0f}, max={X[col].max():.0f}, mean={X[col].mean():.1f}")

print(f"\n✓ Data is ready for modeling!")
print("="*80)

## 🤖 Step 5: Train-Test Split

In [ ]:
print("\n[STEP 5] SPLITTING DATA...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")
print(f"Split ratio: 80-20")

## 🚀 Step 6: Train LightGBM Model

In [ ]:
print("\n[STEP 6] TRAINING LIGHTGBM MODEL...")

# Optimized parameters
lgb_params = {
    'objective': 'regression',
    'metric': 'mape',
    'boosting_type': 'gbdt',
    'num_leaves': 50,
    'max_depth': 8,
    'learning_rate': 0.03,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'min_child_weight': 0.001,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'random_state': 42
}

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

print("Training in progress...")
lgb_model = lgb.train(
    lgb_params,
    train_data,
    num_boost_round=2000,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(200)]
)

print(f"\n✓ Model trained successfully")
print(f"Best iteration: {lgb_model.best_iteration}")

## 📊 Step 7: Evaluate LightGBM Performance

In [ ]:
# Get predictions
y_train_pred = lgb_model.predict(X_train, num_iteration=lgb_model.best_iteration)
y_test_pred = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)

# Calculate metrics
train_mape = mean_absolute_percentage_error(y_train, y_train_pred) * 100
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

test_mape = mean_absolute_percentage_error(y_test, y_test_pred) * 100
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("\n" + "="*80)
print("📊 LIGHTGBM MODEL PERFORMANCE")
print("="*80)

print(f"\n🔵 TRAINING SET:")
print(f"  MAPE: {train_mape:.2f}%")
print(f"  MAE: ₹{train_mae:,.2f}")
print(f"  R²: {train_r2:.4f}")

print(f"\n🟢 TEST SET:")
print(f"  MAPE: {test_mape:.2f}%")
print(f"  MAE: ₹{test_mae:,.2f}")
print(f"  R²: {test_r2:.4f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Train: Actual vs Predicted
axes[0, 0].scatter(y_train, y_train_pred, alpha=0.5, s=10)
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Rent (₹)')
axes[0, 0].set_ylabel('Predicted Rent (₹)')
axes[0, 0].set_title(f'Training: Actual vs Predicted\nR² = {train_r2:.4f}', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Test: Actual vs Predicted
axes[0, 1].scatter(y_test, y_test_pred, alpha=0.5, s=10, color='green')
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Rent (₹)')
axes[0, 1].set_ylabel('Predicted Rent (₹)')
axes[0, 1].set_title(f'Test: Actual vs Predicted\nR² = {test_r2:.4f}', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Residuals
residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

axes[1, 0].scatter(y_train_pred, residuals_train, alpha=0.5, s=10)
axes[1, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Predicted Rent (₹)')
axes[1, 0].set_ylabel('Residuals (₹)')
axes[1, 0].set_title('Training: Residual Plot', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].scatter(y_test_pred, residuals_test, alpha=0.5, s=10, color='green')
axes[1, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Predicted Rent (₹)')
axes[1, 1].set_ylabel('Residuals (₹)')
axes[1, 1].set_title('Test: Residual Plot', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)

## 🧠 Step 8: Train ANN Model

In [ ]:
print("\n[STEP 8] TRAINING ANN MODEL...")

# Prepare data for ANN
X_ann = X.copy()
y_ann = y.copy()

# Encode categorical variables
label_encoders = {}
for col in X_ann.columns:
    if col in ['locality', 'facing']:
        le = LabelEncoder()
        X_ann[col] = le.fit_transform(X_ann[col].astype(str))
        label_encoders[col] = le

# Convert to arrays
X_ann_array = X_ann.values.astype('float32')
y_ann_array = y_ann.values.astype('float32')

# Split
X_train_ann, X_test_ann, y_train_ann, y_test_ann = train_test_split(
    X_ann_array, y_ann_array, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_ann_scaled = scaler.fit_transform(X_train_ann)
X_test_ann_scaled = scaler.transform(X_test_ann)

# Build model
keras.backend.clear_session()

ann_model = keras.Sequential([
    layers.Input(shape=(X_train_ann_scaled.shape[1],)),
    layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    layers.Dense(1, activation='linear')
])

ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='mean_absolute_percentage_error',
    metrics=['mae', 'mse']
)

print("\n📊 ANN Architecture:")
ann_model.summary()

# Callbacks
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=40,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=15,
    min_lr=1e-6
)

# Train
print("\n🚀 Training ANN...")
history = ann_model.fit(
    X_train_ann_scaled,
    y_train_ann,
    validation_data=(X_test_ann_scaled, y_test_ann),
    epochs=400,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=2
)

print("\n✓ ANN training complete!")

## 📊 Step 9: Evaluate ANN Performance

In [ ]:
# Get ANN predictions
y_train_pred_ann = ann_model.predict(X_train_ann_scaled, verbose=0).flatten()
y_test_pred_ann = ann_model.predict(X_test_ann_scaled, verbose=0).flatten()

# Calculate metrics
train_mape_ann = mean_absolute_percentage_error(y_train_ann, y_train_pred_ann) * 100
train_mae_ann = mean_absolute_error(y_train_ann, y_train_pred_ann)
train_r2_ann = r2_score(y_train_ann, y_train_pred_ann)

test_mape_ann = mean_absolute_percentage_error(y_test_ann, y_test_pred_ann) * 100
test_mae_ann = mean_absolute_error(y_test_ann, y_test_pred_ann)
test_r2_ann = r2_score(y_test_ann, y_test_pred_ann)

print("\n" + "="*80)
print("🧠 ANN MODEL PERFORMANCE")
print("="*80)

print(f"\n🔵 TRAINING SET:")
print(f"  MAPE: {train_mape_ann:.2f}%")
print(f"  MAE: ₹{train_mae_ann:,.2f}")
print(f"  R²: {train_r2_ann:.4f}")

print(f"\n🟢 TEST SET:")
print(f"  MAPE: {test_mape_ann:.2f}%")
print(f"  MAE: ₹{test_mae_ann:,.2f}")
print(f"  R²: {test_r2_ann:.4f}")

print("\n" + "="*80)

## 🏆 Step 10: Compare LightGBM vs ANN

In [ ]:
print("\n" + "="*80)
print("🏆 MODEL COMPARISON: LightGBM vs ANN")
print("="*80)

# Create comparison table
print(f"\n{'Model':<15} {'Train MAPE':<15} {'Test MAPE':<15} {'Train R²':<12} {'Test R²':<12}")
print("="*80)
print(f"{'LightGBM':<15} {train_mape:>12.2f}%  {test_mape:>12.2f}%  {train_r2:>10.4f}  {test_r2:>10.4f}")
print(f"{'ANN':<15} {train_mape_ann:>12.2f}%  {test_mape_ann:>12.2f}%  {train_r2_ann:>10.4f}  {test_r2_ann:>10.4f}")
print("="*80)

# Determine winner
if test_mape < test_mape_ann:
    winner = "LightGBM"
    diff = test_mape_ann - test_mape
else:
    winner = "ANN"
    diff = test_mape - test_mape_ann

print(f"\n🏆 Winner: {winner}")
print(f"   Better by: {diff:.2f}% MAPE")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# MAPE Comparison
models = ['LightGBM', 'ANN']
train_mapes = [train_mape, train_mape_ann]
test_mapes = [test_mape, test_mape_ann]
x_pos = np.arange(len(models))
width = 0.35

axes[0, 0].bar(x_pos - width/2, train_mapes, width, label='Train', color='steelblue', alpha=0.8)
axes[0, 0].bar(x_pos + width/2, test_mapes, width, label='Test', color='orange', alpha=0.8)
axes[0, 0].set_ylabel('MAPE (%)')
axes[0, 0].set_title('MAPE Comparison', fontweight='bold')
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(models)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# R² Comparison
train_r2s = [train_r2, train_r2_ann]
test_r2s = [test_r2, test_r2_ann]

axes[0, 1].bar(x_pos - width/2, train_r2s, width, label='Train', color='purple', alpha=0.8)
axes[0, 1].bar(x_pos + width/2, test_r2s, width, label='Test', color='green', alpha=0.8)
axes[0, 1].set_ylabel('R² Score')
axes[0, 1].set_title('R² Score Comparison', fontweight='bold')
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(models)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Training history
axes[1, 0].plot(history.history['loss'], label='Train Loss')
axes[1, 0].plot(history.history['val_loss'], label='Val Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('ANN Training History', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Summary text
axes[1, 1].axis('off')
summary = f"""
🏆 MUMBAI RENT PREDICTION
═══════════════════════════════

📊 Dataset:
  • Total: {len(y):,} properties
  • Train: {len(y_train):,}
  • Test: {len(y_test):,}

🎯 Best Model: {winner}
  • Test MAPE: {min(test_mape, test_mape_ann):.2f}%
  • Test R²: {max(test_r2, test_r2_ann):.4f}
  • Test MAE: ₹{min(test_mae, test_mae_ann):,.0f}

💡 Recommendation:
  Use {winner} for deployment
"""
axes[1, 1].text(0.5, 0.5, summary, transform=axes[1, 1].transAxes,
                fontsize=11, verticalalignment='center', horizontalalignment='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3),
                family='monospace')

plt.tight_layout()
plt.show()

print("\n" + "="*80)

## 🔧 Step 11: Bias Correction & Final MAPE

In [ ]:
print("\n" + "="*80)
print("🔧 BIAS CORRECTION ANALYSIS")
print("="*80)

# Calculate bias for both models
lgb_errors = ((y_test_pred - y_test) / y_test) * 100
ann_errors = ((y_test_pred_ann - y_test_ann) / y_test_ann) * 100

lgb_bias = lgb_errors.mean()
ann_bias = ann_errors.mean()

print(f"\n📊 BIAS ANALYSIS:")
print(f"  LightGBM bias: {lgb_bias:.2f}%")
print(f"  ANN bias: {ann_bias:.2f}%")

# Apply correction to best model
if abs(lgb_bias) < abs(ann_bias) and test_mape < test_mape_ann:
    best_model = "LightGBM"
    bias = lgb_bias
    y_pred_corrected = y_test_pred * (1 - bias/100)
    y_actual = y_test
else:
    best_model = "ANN"
    bias = ann_bias
    y_pred_corrected = y_test_pred_ann * (1 - bias/100)
    y_actual = y_test_ann

correction_factor = 1 - bias/100

# Calculate final metrics
final_mape = mean_absolute_percentage_error(y_actual, y_pred_corrected) * 100
final_mae = mean_absolute_error(y_actual, y_pred_corrected)
final_r2 = r2_score(y_actual, y_pred_corrected)

print(f"\n🏆 FINAL MODEL: {best_model}")
print(f"  Correction factor: {correction_factor:.4f}")
print(f"\n✨ FINAL PERFORMANCE:")
print(f"  Test MAPE: {final_mape:.2f}%")
print(f"  Test MAE: ₹{final_mae:,.2f}")
print(f"  Test R²: {final_r2:.4f}")

print(f"\n💡 DEPLOYMENT FORMULA:")
print(f"  Adjusted_Rent = Predicted_Rent × {correction_factor:.4f}")
print(f"\n  Example:")
print(f"    If model predicts ₹30,000")
print(f"    Adjusted rent = ₹30,000 × {correction_factor:.4f} = ₹{30000 * correction_factor:,.0f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Before correction
y_pred_before = y_test_pred if best_model == "LightGBM" else y_test_pred_ann
axes[0].scatter(y_actual, y_pred_before, alpha=0.5, s=20, color='orange')
axes[0].plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Rent (₹)')
axes[0].set_ylabel('Predicted Rent (₹)')
axes[0].set_title(f'Before Correction\nMAPE: {(test_mape if best_model=="LightGBM" else test_mape_ann):.2f}%', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# After correction
axes[1].scatter(y_actual, y_pred_corrected, alpha=0.5, s=20, color='green')
axes[1].plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Rent (₹)')
axes[1].set_ylabel('Predicted Rent (₹)')
axes[1].set_title(f'After Correction\nMAPE: {final_mape:.2f}%', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ MUMBAI RENT PREDICTION PIPELINE COMPLETE!")
print("="*80)